In [1]:
%%time

import kplex_dev

timeout_replacement = 1800
excel_path = "kplex_ML_data.xlsx"

cat_df, dfs = kplex_dev.load_excel_data(excel_path, timeout_replacement)
cat_df

CPU times: user 1.03 s, sys: 2.12 s, total: 3.15 s
Wall time: 1.01 s


,dataset,n,m,d_avg,d_max,core_max,kcore_list,UMKP,AltRB,kPEX,Pivot_Br,S_Br,k_value
0,MANN-a81,3321.0,5506380.0,1658.0488,3317.0,3280.0,"[(3280,3321)]",1459.5,1800.0,1468.1,1477.4,1504.7,20
1,wikipedia_link_sr,1810.0,1637059.0,904.4525,1809.0,1798.0,"[(1795,1810),(1796,1808),(1797,1804),(1798,1803)]",22.6,18.0,737.7,740.4,738.1,20
2,scc_reality,1332.0,880733.0,661.2110,1331.0,1235.0,"[(1230,1332),(1231,1330),(1232,1309),(1233,126...",1800.0,1214.8,9.4,9.7,9.0,20
3,web-wikipedia_link_en13-all,432.0,93085.0,215.4745,431.0,427.0,"[(427,432)]",239.3,287.1,517.7,504.4,516.7,20
4,livejournal-links,442.0,94435.0,213.6538,441.0,371.0,"[(369,442),(370,409),(371,386)]",1.3,1.6,1.6,1.6,1.6,20
...,...,...,...,...,...,...,...,...,...,...,...,...,...
755,soc-flixster,264.0,11792.0,44.6667,150.0,67.0,"[(52,264),(53,262),(54,261),(55,260),(56,256),...",2.0,0.6,0.5,0.5,2.4,2
756,edit-ukwiki,281.0,14183.0,50.4733,280.0,65.0,"[(40,281),(41,279),(42,275),(43,271),(44,268),...",1.2,1.1,0.6,0.6,0.5,2
757,johnson8-4-4,70.0,1855.0,26.5000,53.0,53.0,"[(53,70)]",0.8,0.9,1.1,1.1,0.3,2
758,hamming6-2,64.0,1824.0,28.5000,57.0,57.0,"[(57,64)]",0.2,0.3,0.2,0.2,0.3,2


# Instance Weight Functions

In [3]:
df = cat_df

In [9]:
# module functions

def time_ratio(vals, epsilon):
    return vals.max() / vals.min() if vals.min() != 0 else vals.max() / epsilon

def time_diff_relative(vals, epsilon): # = time_ratio - 1
    return (vals.max() - vals.min()) / vals.min() if vals.min() != 0 else vals.max() / epsilon

def time_min(vals):
    return vals.min()

In [27]:
# weight schemes
import numpy as np

y_columns = ["UMKP", "AltRB", "kPEX", "Pivot_Br", "S_Br"]
epsilon = 0.001 # need to set !!!

# Basic Schemes
W_TIME_RATIO = 1
W_TIME_DIFF_RELATIVE = 2
W_LOG_TIME_RATIO = 4
W_TIME_MIN = 8
W_LOG_TIME_MIN = 16

# Advanced Schemes
W_TIME_MIN_X_W_LOG_TIME_RATIO = W_TIME_MIN + W_LOG_TIME_RATIO
W_LOG_TIME_MIN_X_W_LOG_TIME_RATIO = W_LOG_TIME_MIN + W_LOG_TIME_RATIO
W_LOG_TIME_MIN_PLUS_W_LOG_TIME_RATIO = (W_LOG_TIME_MIN + W_LOG_TIME_RATIO) * 2

def compute_weight(row, scheme, epsilon):
    vals = row[y_columns]
    if scheme == W_TIME_RATIO:
        return time_ratio(vals, epsilon)
    elif scheme == W_TIME_DIFF_RELATIVE:
        return time_diff_relative(vals, epsilon)
    elif scheme == W_LOG_TIME_RATIO:
        return np.log(time_ratio(vals, epsilon))
    elif scheme == W_TIME_MIN:
        return time_min(vals)
    elif scheme == W_LOG_TIME_MIN:
        return np.log(1 + time_min(vals))
    elif scheme == W_TIME_MIN_X_W_LOG_TIME_RATIO:
        return time_min(vals) * np.log(time_ratio(vals, epsilon))
    elif scheme == W_LOG_TIME_MIN_X_W_LOG_TIME_RATIO:
        return np.log(1 + time_min(vals)) * np.log(time_ratio(vals, epsilon))
    elif scheme == W_LOG_TIME_MIN_PLUS_W_LOG_TIME_RATIO:
        return np.log(1 + time_min(vals)) + np.log(time_ratio(vals, epsilon))
    raise ValueError(f"Unknown scheme: {scheme}")

In [28]:
df["weight"] = df.apply(compute_weight, scheme=W_LOG_TIME_MIN_PLUS_W_LOG_TIME_RATIO, epsilon=epsilon, axis=1)
df

,dataset,n,m,d_avg,d_max,core_max,kcore_list,UMKP,AltRB,kPEX,Pivot_Br,S_Br,k_value,weight
0,MANN-a81,3321.0,5506380.0,1658.0488,3317.0,3280.0,"[(3280,3321)]",1459.5,1800.0,1468.1,1477.4,1504.7,20,7.496227
1,wikipedia_link_sr,1810.0,1637059.0,904.4525,1809.0,1798.0,"[(1795,1810),(1796,1808),(1797,1804),(1798,1803)]",22.6,18.0,737.7,740.4,738.1,20,6.661258
2,scc_reality,1332.0,880733.0,661.2110,1331.0,1235.0,"[(1230,1332),(1231,1330),(1232,1309),(1233,126...",1800.0,1214.8,9.4,9.7,9.0,20,7.600902
3,web-wikipedia_link_en13-all,432.0,93085.0,215.4745,431.0,427.0,"[(427,432)]",239.3,287.1,517.7,504.4,516.7,20,6.253566
4,livejournal-links,442.0,94435.0,213.6538,441.0,371.0,"[(369,442),(370,409),(371,386)]",1.3,1.6,1.6,1.6,1.6,20,1.040548
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
755,soc-flixster,264.0,11792.0,44.6667,150.0,67.0,"[(52,264),(53,262),(54,261),(55,260),(56,256),...",2.0,0.6,0.5,0.5,2.4,2,1.974081
756,edit-ukwiki,281.0,14183.0,50.4733,280.0,65.0,"[(40,281),(41,279),(42,275),(43,271),(44,268),...",1.2,1.1,0.6,0.6,0.5,2,1.280934
757,johnson8-4-4,70.0,1855.0,26.5000,53.0,53.0,"[(53,70)]",0.8,0.9,1.1,1.1,0.3,2,1.561647
758,hamming6-2,64.0,1824.0,28.5000,57.0,57.0,"[(57,64)]",0.2,0.3,0.2,0.2,0.3,2,0.587787


In [29]:
def assign_weights(df, scheme, epsilon):
    df["weight"] = df.apply(compute_weight, scheme=scheme, epsilon=epsilon, axis=1)
    
assign_weights(df, W_LOG_TIME_MIN_PLUS_W_LOG_TIME_RATIO, epsilon)
df

,dataset,n,m,d_avg,d_max,core_max,kcore_list,UMKP,AltRB,kPEX,Pivot_Br,S_Br,k_value,weight
0,MANN-a81,3321.0,5506380.0,1658.0488,3317.0,3280.0,"[(3280,3321)]",1459.5,1800.0,1468.1,1477.4,1504.7,20,7.496227
1,wikipedia_link_sr,1810.0,1637059.0,904.4525,1809.0,1798.0,"[(1795,1810),(1796,1808),(1797,1804),(1798,1803)]",22.6,18.0,737.7,740.4,738.1,20,6.661258
2,scc_reality,1332.0,880733.0,661.2110,1331.0,1235.0,"[(1230,1332),(1231,1330),(1232,1309),(1233,126...",1800.0,1214.8,9.4,9.7,9.0,20,7.600902
3,web-wikipedia_link_en13-all,432.0,93085.0,215.4745,431.0,427.0,"[(427,432)]",239.3,287.1,517.7,504.4,516.7,20,6.253566
4,livejournal-links,442.0,94435.0,213.6538,441.0,371.0,"[(369,442),(370,409),(371,386)]",1.3,1.6,1.6,1.6,1.6,20,1.040548
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
755,soc-flixster,264.0,11792.0,44.6667,150.0,67.0,"[(52,264),(53,262),(54,261),(55,260),(56,256),...",2.0,0.6,0.5,0.5,2.4,2,1.974081
756,edit-ukwiki,281.0,14183.0,50.4733,280.0,65.0,"[(40,281),(41,279),(42,275),(43,271),(44,268),...",1.2,1.1,0.6,0.6,0.5,2,1.280934
757,johnson8-4-4,70.0,1855.0,26.5000,53.0,53.0,"[(53,70)]",0.8,0.9,1.1,1.1,0.3,2,1.561647
758,hamming6-2,64.0,1824.0,28.5000,57.0,57.0,"[(57,64)]",0.2,0.3,0.2,0.2,0.3,2,0.587787


In [37]:
df_sorted = df.sort_values(by="weight", ascending=False).reset_index(drop=True)
df_sorted

,dataset,n,m,d_avg,d_max,core_max,kcore_list,UMKP,AltRB,kPEX,Pivot_Br,S_Br,k_value,weight
0,edit-nlwiki,1560.0,192960.0,123.6923,1520.0,157.0,"[(70,1560),(71,1559),(72,1557),(73,1556),(74,1...",272.0,519.1,39.5,41.0,1800.0,7,14.136165
1,edit-viwiki,1552.0,58983.0,38.0045,1477.0,54.0,"[(27,1552),(28,1491),(29,1414),(30,1344),(31,1...",1404.9,115.0,39.3,42.6,1800.0,15,14.136021
2,ia-wiki-Talk-dir,844.0,83842.0,99.3389,626.0,125.0,"[(69,844),(70,843),(71,841),(72,840),(73,836),...",112.6,175.4,38.6,44.3,1800.0,10,14.135127
3,link-dynamic-dewiki,844.0,122737.0,145.4230,664.0,209.0,"[(162,844),(163,738),(164,736),(165,733),(166,...",1800.0,1800.0,38.5,38.3,38.5,15,14.134552
4,wiki-Talk,844.0,83842.0,99.3389,626.0,125.0,"[(69,844),(70,843),(71,841),(72,840),(73,836),...",113.1,174.2,37.9,44.7,1800.0,10,14.133601
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
858,brock200-2,200.0,9876.0,49.3800,114.0,84.0,"[(80,200),(81,199),(82,198),(83,197),(84,195)]",1800.0,1800.0,1800.0,1800.0,1800.0,8,0.000000
859,p-hat500-1,500.0,31565.0,63.1300,204.0,86.0,"[(58,500),(59,499),(60,498),(61,497),(62,496),...",1800.0,1800.0,1800.0,1800.0,1800.0,8,0.000000
860,aff-orkut-user2groups,802219.0,67140962.0,83.6941,275825.0,273.0,"[(12,802219),(13,786112),(14,770790),(15,75655...",1800.0,1800.0,1800.0,1800.0,1800.0,10,0.000000
861,keller4,171.0,9435.0,55.1754,124.0,102.0,"[(102,171)]",1800.0,1800.0,1800.0,1800.0,1800.0,8,0.000000


In [14]:
df_sorted.to_csv("sorted_data.csv", index=False)

In [30]:
def export_sorted(df, output_file = "sorted_data.csv"):
    df_sorted = df.sort_values(by="weight", ascending=False).reset_index(drop=True)
    df_sorted.to_csv(output_file, index=False)
    
export_sorted(df)

In [2]:
# test run

import kplex_dev

epsilon = 0.001 # need to set !!!

kplex_dev.assign_weights(cat_df, kplex_dev.W_LOG_TIME_RATIO, epsilon)
display(cat_df)
kplex_dev.export_sorted(cat_df)

,dataset,n,m,d_avg,d_max,core_max,kcore_list,UMKP,AltRB,kPEX,Pivot_Br,S_Br,k_value,weight
0,MANN-a81,3321.0,5506380.0,1658.0488,3317.0,3280.0,"[(3280,3321)]",1459.5,1800.0,1468.1,1477.4,1504.7,20,0.209693
1,wikipedia_link_sr,1810.0,1637059.0,904.4525,1809.0,1798.0,"[(1795,1810),(1796,1808),(1797,1804),(1798,1803)]",22.6,18.0,737.7,740.4,738.1,20,3.716819
2,scc_reality,1332.0,880733.0,661.2110,1331.0,1235.0,"[(1230,1332),(1231,1330),(1232,1309),(1233,126...",1800.0,1214.8,9.4,9.7,9.0,20,5.298317
3,web-wikipedia_link_en13-all,432.0,93085.0,215.4745,431.0,427.0,"[(427,432)]",239.3,287.1,517.7,504.4,516.7,20,0.771678
4,livejournal-links,442.0,94435.0,213.6538,441.0,371.0,"[(369,442),(370,409),(371,386)]",1.3,1.6,1.6,1.6,1.6,20,0.207639
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
755,soc-flixster,264.0,11792.0,44.6667,150.0,67.0,"[(52,264),(53,262),(54,261),(55,260),(56,256),...",2.0,0.6,0.5,0.5,2.4,2,1.568616
756,edit-ukwiki,281.0,14183.0,50.4733,280.0,65.0,"[(40,281),(41,279),(42,275),(43,271),(44,268),...",1.2,1.1,0.6,0.6,0.5,2,0.875469
757,johnson8-4-4,70.0,1855.0,26.5000,53.0,53.0,"[(53,70)]",0.8,0.9,1.1,1.1,0.3,2,1.299283
758,hamming6-2,64.0,1824.0,28.5000,57.0,57.0,"[(57,64)]",0.2,0.3,0.2,0.2,0.3,2,0.405465
